[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fried-toofuu/music-data/blob/main/data_pipeline/notebooks/02_feature_extraction_demo.ipynb)

# Trình diễn Trích xuất Đặc trưng (Feature Engineering)
Notebook này mô tả trực quan quá trình xử lý 1 file âm thanh từ HF Datasets: từ Waveform gốc, chia nhỏ (segmentation) đến việc chuyển đổi thành Mel-Spectrogram qua torchaudio.


In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

# Xác định môi trường
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    SRC_DIR = '/content/music-data/data_pipeline'
else:
    from pathlib import Path
    SRC_DIR = str(Path().resolve().parent)

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)


## 1. Khởi tạo Pipeline AudioTransform
Tạo một instance của `AudioTransform` từ codebase (chứa logic resample, extract segment và mel-spectrogram).


In [ ]:
from src.features.audio_transforms import AudioTransform

# Khởi tạo thông số chuẩn của Project
transform = AudioTransform(
    sample_rate=32000,
    duration=3.0,
    segment_overlap=0.0,
    n_mels=128,
    n_fft=1024,
    hop_length=512,
    f_min=0,
    f_max=16000,
    norm_type="slaney",
    target_frames=188,
    log_epsilon=1e-5,
    device=torch.device('cpu')
)


## 2. Tải 1 Sample từ Hugging Face


In [1]:
print("Đang lấy 1 sample âm thanh từ HF (Streaming)...")
ds_stream = load_dataset('Khahn-nh/fma-small-1', split='train', streaming=True)
first_track = next(iter(ds_stream))

# Trong HF, first_track['audio'] trả về 1 dict: {'path': ..., 'array': ..., 'sampling_rate': ...}
audio_array = first_track['audio']['array']
sr = first_track['audio']['sampling_rate']

print(f"Tên/ID bài hát: {first_track.get('track_id', 'Unknown')}")
print(f"Thể loại: {first_track.get('label', 'Unknown')}")
print(f"Original Sample Rate: {sr}")

# Visualize array
plt.figure(figsize=(10, 3))
plt.plot(np.linspace(0, len(audio_array)/sr, len(audio_array)), audio_array)
plt.title('Waveform file gốc (trực tiếp từ mảng numpy)')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.show()


Đang lấy 1 sample âm thanh từ HF (Streaming)...


NameError: name 'load_dataset' is not defined

## 3. Chuyển đổi Segment thành Mel-Spectrogram
Do luồng HF trả về mảng numpy, chúng ta chuyển nó sang shape Tensor mà hàm `transform` yêu cầu `(1, N)`


In [2]:
import torchaudio.transforms as T

waveform = torch.from_numpy(audio_array).float().unsqueeze(0) # (1, N)

# Resample thủ công (Bypass PyAV load step)
if sr != transform.sample_rate:
    resampler = T.Resample(orig_freq=sr, new_freq=transform.sample_rate)
    waveform_resampled = resampler(waveform)
else:
    waveform_resampled = waveform

print(f"Waveform sau khi resample: {waveform_resampled.shape}")

try:
    # Lấy phân đoạn đầu tiên và cho ra Mel Spec
    segments = transform.extract_segments(waveform_resampled)
    mel_spec = transform.to_mel_spectrogram(segments[0])
    
    print(f"Kích thước Mel Spectrogram: {mel_spec.shape} -> (channels, n_mels, frames)")
    
    # Dùng matplotlib cơ bản vẽ.
    plt.figure(figsize=(10, 4))
    plt.imshow(mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Log-Mel Spectrogram (Segment 0-3s)')
    plt.xlabel('Frames')
    plt.ylabel('Mel bins')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Lỗi phát sinh:", e)


NameError: name 'torch' is not defined